
# 07 · Genie Agent — natural-language access, validated
The Genie Agent *Northstar Network Optimization Genie* (space-as-code in `genie/build_space.py`) sits on the gold,
ML-score and GenAI tables. This task asks it the business questions a logistics director and a finance partner
would ask, prints the SQL Genie generated and the result rows, and checks two answers against hand-written SQL.

In [ ]:
dbutils.widgets.text("catalog", "cpg_demos")
dbutils.widgets.text("genie_space_id", "")
CAT = dbutils.widgets.get("catalog")
SPACE = dbutils.widgets.get("genie_space_id")
G = f"{CAT}.netopt_gold"
assert SPACE, "genie_space_id parameter is required"

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()


def ask(question: str):
    msg = w.genie.start_conversation_and_wait(SPACE, question)
    out = {"question": question, "status": msg.status.value if msg.status else None, "text": [], "sql": None, "columns": [], "rows": []}
    for att in msg.attachments or []:
        if att.text and att.text.content:
            out["text"].append(att.text.content)
        if att.query:
            out["sql"] = att.query.query
            if att.query.description:
                out["text"].append(att.query.description)
            res = w.genie.get_message_attachment_query_result(SPACE, msg.conversation_id, msg.message_id, att.attachment_id)
            sr = res.statement_response
            if sr and sr.manifest and sr.manifest.schema:
                out["columns"] = [c.name for c in sr.manifest.schema.columns]
                out["rows"] = (sr.result.data_array or []) if sr.result else []
    return out


def show(r, max_rows=10):
    print("=" * 110)
    print(f"Q: {r['question']}   [status={r['status']}]")
    for t in r["text"]:
        print(f"A: {t}")
    if r["sql"]:
        print("SQL generated by Genie:\n  " + r["sql"].strip().replace("\n", "\n  "))
    if r["columns"]:
        print("Result:", " | ".join(r["columns"]))
        for row in r["rows"][:max_rows]:
            print("   ", " | ".join("" if v is None else str(v) for v in row))
        if len(r["rows"]) > max_rows:
            print(f"    ... {len(r['rows'])} rows")

In [ ]:
QUESTIONS = [
    "How much did out-of-route freight cost us in excess of plan over the last 3 months?",
    "Which 10 lanes have the highest expected monthly leakage?",
    "Which product families drive the most costly out-of-route moves?",
    "What actions are recommended for the high risk lanes?",
    "What share of shipped cases moved out of route in the latest month?",
    "How many reconstructed routes happened but were not planned, and what was their total cost impact?",
]
results = []
for q in QUESTIONS:
    r = ask(q)
    results.append(r)
    show(r)

Q: How much did out-of-route freight cost us in excess of plan over the last 3 months?   [status=COMPLETED]
A: You want to know the total excess cost of out-of-route freight compared to the plan for the last three months.
A: Out-of-route freight cost **$6,481,690** in excess of plan over the last 3 months. Based on the result shown, the 3-month total excess cost is **$6,481,690**.
SQL generated by Genie:
  WITH latest_month AS (
    SELECT MAX(to_date(concat(`year_month`, '-01'))) AS `max_month`
    FROM `cpg_demos`.`netopt_gold`.`oor_analysis`
    WHERE `year_month` IS NOT NULL
  ), trailing_3 AS (
    SELECT
      `year_month`,
      `total_cost_impact`
    FROM `cpg_demos`.`netopt_gold`.`oor_analysis`, latest_month
    WHERE `drill_tab` = 'costly_oor'
      AND `year_month` IS NOT NULL
      AND to_date(concat(`year_month`, '-01')) BETWEEN add_months(`max_month`, -2) AND `max_month`
  )
  SELECT ROUND(SUM(`total_cost_impact`), 0) AS `excess_cost_usd`
  FROM trailing_3
Result: excess

## Answer checks against hand-written SQL

In [ ]:
def first_number(r):
    for row in r["rows"]:
        for v in row:
            try:
                return float(v)
            except (TypeError, ValueError):
                continue
    return None


expected_high = spark.sql(f"SELECT count(*) FROM {G}.lane_action_queue WHERE risk_tier = 'HIGH'").first()[0]
r = ask("How many HIGH risk lanes are in the action queue? Return just the count.")
show(r)
print(f"CHECK high-risk lane count: genie={first_number(r)} expected={expected_high} -> "
      f"{'PASS' if first_number(r) == expected_high else 'MISMATCH'}")

expected_2026 = spark.sql(f"""SELECT round(sum(total_cost_impact), 0) FROM {G}.oor_analysis
                              WHERE drill_tab = 'costly_oor' AND year_month >= '2026-01'""").first()[0]
r = ask("What was the total costly out-of-route leakage in 2026 (year_month from 2026-01)? Return one number in USD.")
show(r)
g = first_number(r)
ok = g is not None and abs(g - float(expected_2026)) <= max(1.0, 0.001 * float(expected_2026))
print(f"CHECK 2026 leakage: genie={g} expected={expected_2026} -> {'PASS' if ok else 'MISMATCH'}")

Q: How many HIGH risk lanes are in the action queue? Return just the count.   [status=COMPLETED]
A: You want to know the total number of lanes classified as high risk in the action queue.
A: **12** HIGH risk lanes are in the action queue.
SQL generated by Genie:
  SELECT COUNT(*) AS `high_risk_lane_count`
  FROM `cpg_demos`.`netopt_gold`.`lane_action_queue`
  WHERE `risk_tier` = 'HIGH'
Result: high_risk_lane_count
    12
CHECK high-risk lane count: genie=12.0 expected=12 -> PASS
Q: What was the total costly out-of-route leakage in 2026 (year_month from 2026-01)? Return one number in USD.   [status=COMPLETED]
A: You want to see the total costly out-of-route freight leakage in US dollars starting from January 2026 onward as a single total number.
A: The total costly out-of-route leakage for **year_month >= 2026-01** is **$8,494,718 USD**. Based on the single result returned, **$8,494,718 USD** is the full total captured by this query for 2026-to-date.
SQL generated by Genie:
  SELECT ROU